[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P12/P12-1_Optimiser.ipynb)

# Résolution numérique et optimisation

Chapitre 12 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre bureau d'études doit dimensionner une poutre qui fléchit de moins de deux millimètres avec le moins d'acier possible, un arbre qui tient son couple, un plan de charge au moindre coût, une loi de fatigue qui passe au mieux par des essais dispersés ? Dans chaque cas, la formule existe dans un sens et pas dans l'autre, ou plusieurs exigences se disputent la même solution. Vous voudriez que l'ordinateur cherche, et qu'il dise si sa réponse est sûre ?

**Objectif terminal.** Ce chapitre montre comment, avec le module `scipy.optimize` de SciPy.

**Objectif instrumental.**

- Résoudre une équation avec `brentq`, un système avec `root`.
- Minimiser sous contraintes avec `minimize` et interpréter le multiplicateur de Lagrange.
- Poser un plan de charge en programme linéaire et lire ses prix.
- Ajuster une loi avec `curve_fit` et chiffrer son incertitude.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 3 à 5 et 10, après une filière scientifique.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 12.1 | Diamètre d'un arbre par `brentq` | 10 min |
| 12.2 | Section optimale d'une poutre | 20 min |
| 12.3 | Deux références sur deux machines | 15 min |
| 12.4 | Exposant de Basquin et son incertitude (défi) | 25 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("scipy", version("scipy"))
print("sympy", version("sympy"))

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [ ]:
import numpy as np
import sympy as sp
from scipy.optimize import brentq, curve_fit, linprog, minimize, root

### Section 12.1 : une équation se résout numériquement avec brentq

---------------------------------------------------------------------------------------------------------

In [ ]:
def sigma_max_MPa(d_mm, W_mm=50.0, e_mm=5.0, effort_N=20000.0):
    """Contrainte maximale au bord d'un trou central (Heywood)."""
    kt = 2 + (1 - d_mm / W_mm) ** 3
    return kt * effort_N / ((W_mm - d_mm) * e_mm)

def ecart_MPa(d_mm):
    return sigma_max_MPa(d_mm) - 300.0       # nul à la limite de 300 MPa

d_max_mm = brentq(ecart_MPa, 10.0, 40.0)     # zéro de g entre 10 et 40 mm
print(f"trou {d_max_mm:.2f} mm, contrainte {sigma_max_MPa(d_max_mm):.1f} MPa")
# seconde voie : dichotomie (chapitre 3), 60 tours sur [10 ; 40] mm
a, b = 10.0, 40.0

for _ in range(60):
    milieu = (a + b) / 2
    if ecart_MPa(milieu) < 0:
        a = milieu
    else:
        b = milieu

In [ ]:
assert abs((a + b) / 2 - d_max_mm) < 1e-6, "brentq doit retrouver la dichotomie"
assert ecart_MPa(10.0) < 0 < ecart_MPa(40.0), "g change de signe entre 10 et 40 mm"
contrainte_dix = sigma_max_MPa(10.0)
print(f"contrôle : contrainte du trou de 10 mm = {contrainte_dix:.1f} MPa (chapitre 3 : 251,2 MPa)")
assert abs(contrainte_dix - 251.2) < 0.05
grille = np.linspace(1.0, 49.0, 25)           # balayage grossier
signes = np.sign([ecart_MPa(d) for d in grille])
k = np.where(np.diff(signes) != 0)[0]
print("le signe change entre", grille[k], "et", grille[k + 1])
d_mm, bilan = brentq(ecart_MPa, 19.0, 21.0, xtol=1e-3, full_output=True)
print(f"{d_mm:.3f} mm, convergé : {bilan.converged}")
print(f"{bilan.iterations} itérations, {bilan.function_calls} évaluations")

try:
    brentq(ecart_MPa, 10.0, 20.0)             # même signe aux deux bornes
except ValueError as erreur:
    print(erreur)

In [ ]:
assert bilan.converged and len(k) == 1 and abs(d_mm - d_max_mm) < 1e-3
F_N, L_mm, E_MPa = 2000.0, 1000.0, 210000.0   # poutre du chapitre 3

def fleche_mm(b_mm, h_mm):
    return F_N * L_mm**3 / (4 * E_MPa * b_mm * h_mm**3)

def sigma_MPa(b_mm, h_mm):
    return 3 * F_N * L_mm / (2 * b_mm * h_mm**2)

print(f"20 x 40 mm : {fleche_mm(20.0, 40.0):.4f} mm,",
      f"{sigma_MPa(20.0, 40.0):.1f} MPa")

def residus(x):
    b_mm, h_mm = x
    return [fleche_mm(b_mm, h_mm) - 2.0, sigma_MPa(b_mm, h_mm) - 160.0]

solution = root(residus, x0=[10.0, 40.0])
print(solution.x, solution.success)
print(f"résidu maximal : {max(abs(solution.fun)):.1e},",
      f"{solution.nfev} évaluations")

In [ ]:
# seconde voie : division des deux équations, h = L² sigma / (6 E f), puis b = Ks / h²
f_max_mm, sigma_max_adm = 2.0, 160.0
K_f = F_N * L_mm**3 / (4 * E_MPa * f_max_mm)       # b h³ = K_f
K_s = 3 * F_N * L_mm / (2 * sigma_max_adm)         # b h² = K_s
h_formule = L_mm**2 * sigma_max_adm / (6 * E_MPa * f_max_mm)
b_formule = K_s / h_formule**2
assert solution.success and abs(h_formule - K_f / K_s) < 1e-9
assert abs(solution.x[0] - b_formule) < 1e-6 and abs(solution.x[1] - h_formule) < 1e-6

for depart in ([1.0, 1.0], [50.0, 100.0], [0.5, 200.0]):          # un autre départ, la même solution
    autre = root(residus, x0=depart)
    assert autre.success and np.allclose(autre.x, solution.x, atol=1e-6), depart

## Exercice 12.1 : Diamètre d'un arbre par `brentq` (10 min)

Retrouvez le diamètre de l'arbre du chapitre 3, qui transmet $500$ N$\cdot$m sans dépasser $60$ MPa de cisaillement, avec `brentq` entre $10$ et $100$ mm. Réglez `xtol` à $0{,}005$ mm, la précision de la dichotomie, et comparez le nombre d'évaluations à celui de la dichotomie.

**Résultat attendu.** Diamètre de 34,88 mm à $0{,}005$ mm près ; 10 évaluations, contre 15 tours de dichotomie.

**Seconde voie.** La formule fermée $d = \bigl(16\,T/(\pi\,\tau)\bigr)^{1/3}$ donne 34,8816 mm.

**Indication.** Écrivez le cisaillement moins $60$ en fonction du diamètre : positif à $10$ mm, négatif à $100$ mm, il est encadré. Réglez `xtol` à $0{,}005$, demandez `full_output` et lisez `function_calls`. La dichotomie coupe toujours en deux ; `brentq` gagne quelques évaluations.

Les contrôles de la cellule lisent les variables `bilan_arbre`, `d_arbre_mm`, `d_formule_mm`, `tours` : gardez ces noms.

In [ ]:
T_Nm, tau_adm_MPa = 500.0, 60.0
# Votre code :
assert abs(d_arbre_mm - d_formule_mm) < 0.005
# Votre code :
assert tours == 15 and bilan_arbre.function_calls < tours
print("Exercice 12.1 : contrôles passés")

### Section 12.2 : minimiser sous contrainte

---------------------------------------------------------------------------------------------------------

In [ ]:
D_mm = 50.0                                   # diamètre du rond
rng = np.random.default_rng(42)
m = 2500                                      # nombre de sections tirées
b_rand = rng.uniform(1.0, D_mm, m)
h_rand = rng.uniform(1.0, D_mm, m)
tient = b_rand**2 + h_rand**2 <= D_mm**2     # masque : tient dans le rond
vals = np.where(tient, fleche_mm(b_rand, h_rand), np.inf)
print(f"{tient.sum()} sections admissibles,",
      f"la meilleure donne {vals.min():.3f} mm")
f_hasard_mm = vals.min()
n_admissibles = int(tient.sum())
b, D = sp.symbols("b D", positive=True)
h = sp.sqrt(D**2 - b**2)                      # g = 0 résolue en h
b_opt = sp.solve(sp.diff(b * h**3, b), b)[0]  # f minimale : b h³ maximal
print(b_opt, sp.simplify(h.subs(b, b_opt) / b_opt))
assert b_opt == D / 2 and sp.simplify(h.subs(b, b_opt) / b_opt) == sp.sqrt(3)

In [ ]:
K = fleche_mm(1.0, 1.0)                       # f = K / (b h³)

def lagrange(z):
    b, h, lam = z
    grad_f = np.array([-K / (b**2 * h**3), -3 * K / (b * h**4)])
    grad_g = np.array([2 * b, 2 * h])
    delta = grad_f - lam * grad_g
    return [delta[0], delta[1], b**2 + h**2 - D_mm**2]

essai = root(lagrange, x0=[40.0, 30.0, -1e-3])
print(essai.success, essai.message)
assert not essai.success, "le système brut doit échouer (leçon d'échelle du texte)"

def lagrange_ordre_un(z):
    b, h, mu = z                              # mu = 1000 lam, d'ordre un
    r = lagrange([b, h, mu / 1000])
    return [r[0] * 100, r[1] * 100, r[2] / D_mm**2]

bon = root(lagrange_ordre_un, x0=[40.0, 30.0, -1.0])
b_L, h_L, mu = bon.x

In [ ]:
print(f"b = {b_L:.2f} mm, h = {h_L:.2f} mm, lam = {mu / 1000:.3e} 1/mm")
print(bon.success, bon.nfev)
assert bon.success
lam_mm = mu / 1000
# Objectif et contrainte géométrique
def objectif(x):
    return fleche_mm(x[0], x[1])

def contrainte(x):
    return x[0]**2 + x[1]**2 - D_mm**2

# Résolution par la méthode SLSQP
rond = {"type": "eq", "fun": contrainte}
res = minimize(objectif, x0=[40.0, 30.0], method="SLSQP", constraints=[rond])
print(res.x.round(3), res.success)
print(f"flèche {res.fun:.4f} mm, {res.nit} itérations,",
      f"{res.nfev} évaluations")
assert res.success
assert f_hasard_mm >= res.fun - 1e-9 and res.nfev < 2500
# seconde voie : le multiplicateur par moindres carrés, grad f = lam grad g au point trouvé
bs, hs = res.x
grad_f = np.array([-K / (bs**2 * hs**3), -3 * K / (bs * hs**4)])
grad_g = np.array([2 * bs, 2 * hs])

In [ ]:
lam_formule = grad_f @ grad_g / (grad_g @ grad_g)
# sensibilité : agrandir le rond de dD change la flèche minimale de 2 D lam dD au premier ordre
sensibilite = 2 * D_mm * lam_mm
f_etoile = lambda D_: 16 * K / (3 * 3**0.5 * D_**4)     # flèche minimale pour un rond de diamètre D_
assert abs(f_etoile(D_mm) - res.fun) < 1e-6
assert abs((f_etoile(D_mm + 1e-4) - f_etoile(D_mm)) / 1e-4 - sensibilite) < 1e-4
print(f"contrôle : sensibilité {sensibilite:.4f} mm par mm de diamètre ; différence finie sur 1 mm "
      f"{f_etoile(D_mm + 1) - f_etoile(D_mm):.4f} mm")
rond_ineq = {"type": "ineq", "fun": lambda x: D_mm**2 - x[0]**2 - x[1]**2}
res2 = minimize(objectif, x0=[10.0, 10.0], method="SLSQP",
                bounds=[(1.0, D_mm)] * 2, constraints=[rond_ineq])
print(res2.x.round(3), res2.nit)
assert res2.success and abs(res2.x[0] - 25.0) < 1e-3 and abs(res2.x[1] - 43.30127) < 1e-3
assert abs(res2.x[0]**2 + res2.x[1]**2 - D_mm**2) < 1e-3, "inégalité active"

## Exercice 12.2 : Section optimale d'une poutre (20 min)

La poutre précédente ($F = 2$ kN, $L = 1$ m, acier $E = 210\,000$ MPa) doit avoir la plus petite aire $b\,h$ sous trois contraintes (figure du chapitre) : une flèche d'au plus $L/500$, une contrainte d'au plus $160$ MPa, et une hauteur d'au plus quatre fois la largeur, règle de proportion qui limite le risque de déversement sans dispenser de le vérifier. Écrivez les trois contraintes sous la forme $c(x) \ge 0$ et lancez `minimize` depuis $20 \times 40$ mm. Lesquelles sont actives ?

**Résultat attendu.** Section de 11,68 mm sur 46,71 mm (à $0{,}005$ mm près), aire de 545,5 mm² (à $0{,}05$ mm² près) ; contrainte de 117,7 MPa, donc inactive.

**Seconde voie.** Flèche et rapport $h = 4b$ sont actifs : $b = \bigl(F L^3/(256\,E\,f_{\max})\bigr)^{1/4}$ donne 11,678 mm.

**Indication.** Les trois contraintes s'écrivent $2 - f(b, h) \ge 0$, $160 - \sigma(b, h) \ge 0$ et $4b - h \ge 0$, chacune dans un dictionnaire `"ineq"`. L'objectif est $b\,h$. À l'optimum, la flèche vaut $2$ mm et $h = 4b$ : ces deux contraintes sont actives ; celle de $160$ MPa ne l'est pas.

Les contrôles de la cellule lisent les variables `b_coin`, `b_s`, `h_s`, `res_s`, `sigma_s` : gardez ces noms.

In [ ]:
# Votre code :
assert res_s.success
# Votre code :
assert abs(b_s - b_coin) < 0.005 and abs(h_s - 4 * b_coin) < 0.02
assert sigma_s < sigma_max_adm, "la contrainte de 160 MPa n'est pas active"
print("Exercice 12.2 : contrôles passés")

### Section 12.3 : un plan de charge est un programme linéaire

---------------------------------------------------------------------------------------------------------

In [ ]:
cout = [85.0, 45.0]                  # euros par heure : rapide, lent
A_ub = [[-12.0, -8.0],               # pièces >= 1000, signe changé
        [1.0, 1.0]]                  # heures d'opérateur <= 105
b_ub = [-1000.0, 105.0]
plan = linprog(cout, A_ub=A_ub, b_ub=b_ub, bounds=[(0, 60), (0, 80)])
print(plan.status, plan.message)
print("heures :", plan.x, "coût :", plan.fun)
print("jeu des contraintes :", plan.slack)
assert plan.status == 0 and np.allclose(plan.x, [40.0, 65.0]) and abs(plan.fun - 6325.0) < 1e-6
assert np.allclose(plan.slack, 0.0), "commande et opérateurs sont saturés"
# seconde voie : intersection de la demande et de la limite d'opérateurs, par np.linalg.solve
sommet = np.linalg.solve([[12.0, 8.0], [1.0, 1.0]], [1000.0, 105.0])
assert np.allclose(sommet, plan.x)
# le raisonnement glouton échoue : centre lent plein (80 h), le reste sur le rapide
heures_lent = 80.0
heures_rapide = (1000.0 - 8.0 * heures_lent) / 12.0

In [ ]:
assert 45.0 / 8.0 < 85.0 / 12.0 and heures_rapide + heures_lent > 105.0
prix = plan.ineqlin.marginals       # variation du coût par unité de b_ub
print(f"une pièce de plus coûte {-prix[0]:.1f} euros")
print(f"une heure d'opérateur de plus fait gagner {-prix[1]:.1f} euros")
plan_plus = linprog(cout, A_ub=A_ub, b_ub=[-1000.0, 106.0],
                    bounds=[(0, 60), (0, 80)])
print(f"avec 106 h : {plan_plus.fun - plan.fun:.1f} euros")
assert abs(-prix[0] - 10.0) < 1e-9 and abs(-prix[1] - 35.0) < 1e-9
assert abs((plan_plus.fun - plan.fun) - prix[1]) < 1e-9
plan_piece = linprog(cout, A_ub=A_ub, b_ub=[-1001.0, 105.0], bounds=[(0, 60), (0, 80)])
assert abs((plan_piece.fun - plan.fun) + prix[0]) < 1e-9
infaisable = linprog(cout, A_ub=A_ub, b_ub=[-1000.0, 70.0], bounds=[(0, 60), (0, 80)])
non_borne = linprog([-1.0, -1.0])
assert infaisable.status == 2 and non_borne.status == 3, "statuts 2 et 3 cités dans le texte"

## Exercice 12.3 : Deux références sur deux machines (15 min)

Un atelier fabrique deux références, A et B, qui dégagent une marge de $38$ et $48$ euros par pièce. La machine 1 y consacre $0{,}6$ h par pièce A et $0{,}9$ h par pièce B, et dispose de $360$ h ; la machine 2, $0{,}8$ h et $0{,}3$ h, pour $300$ h. Quel plan maximise la marge ? Lisez ensuite ce que vaut une heure de plus sur chaque machine.

**Résultat attendu.** 300 pièces A et 200 pièces B, marge de 21 000 euros ; une heure de plus vaut 50 euros sur la machine 1 et 10 euros sur la machine 2.

**Seconde voie.** Les deux machines sont saturées : `np.linalg.solve` sur les deux équations d'heures redonne le plan (section 11.1).

**Indication.** Maximiser la marge, c'est minimiser son opposé : `c = [-38, -48]`. Les heures par pièce forment la matrice `A_ub = [[0.6, 0.9], [0.8, 0.3]]`, les heures disponibles le vecteur `b_ub = [360, 300]` ; les bornes par défaut vont de zéro à l'infini. Les prix duaux se lisent avec leur signe changé.

Les contrôles de la cellule lisent les variables `charge`, `heures_dispo`, `heures_par_piece`, `marge_totale`, `pieces`, `prix_heure` : gardez ces noms.

In [ ]:
# Votre code :
assert charge.status == 0 and np.allclose(pieces, [300.0, 200.0])
assert abs(marge_totale - 21000.0) < 1e-6
assert np.allclose(prix_heure, [50.0, 10.0])
assert np.allclose(np.linalg.solve(heures_par_piece, heures_dispo), pieces)
assert abs(prix_heure @ heures_dispo - marge_totale) < 1e-6
# Votre code :
print("Exercice 12.3 : contrôles passés")

### Section 12.4 : ajuster une loi aux mesures (défi de Basquin)

---------------------------------------------------------------------------------------------------------

In [ ]:
def campagne(graine):
    """Une campagne simulée : 5 niveaux de contrainte, 3 éprouvettes par niveau, bruit multiplicatif."""
    generateur = np.random.default_rng(graine)
    S = np.repeat([150.0, 200.0, 250.0, 300.0, 350.0], 3)
    bruit = generateur.normal(0.0, 0.15, S.size)
    return S, 14.0 - 3.5 * np.log10(S) + bruit

rng = np.random.default_rng(42)
niveaux_MPa = [150.0, 200.0, 250.0, 300.0, 350.0]       # 5 niveaux
S_MPa = np.repeat(niveaux_MPa, 3)                       # 3 éprouvettes chacun
m_vrai, logC_vrai, dispersion = 3.5, 14.0, 0.15
bruit = rng.normal(0.0, dispersion, S_MPa.size)
logN = logC_vrai - m_vrai * np.log10(S_MPa) + bruit
N = 10**logN                                            # cycles à la rupture
print(S_MPa.size, "essais, de", int(N.min()), "à", int(N.max()), "cycles")
pente, ordonnee = np.polyfit(np.log10(S_MPa), logN, 1)

In [ ]:
print(f"droite des moindres carrés : m = {-pente:.3f},",
      f"log C = {ordonnee:.2f}")
assert np.allclose(campagne(42)[1], logN), "campagne() rejoue la graine 42 du texte"

def wohler(S, logC, m):
    return logC - m * np.log10(S)             # log10 de la durée de vie

popt, pcov = curve_fit(wohler, S_MPa, logN, p0=[14.0, 3.0])
erreurs = np.sqrt(np.diag(pcov))              # écarts-types des paramètres
m_bas, m_haut = popt[1] - 2 * erreurs[1], popt[1] + 2 * erreurs[1]
print(f"m = {popt[1]:.3f} +/- {erreurs[1]:.3f}")
print(f"intervalle à 2 écarts-types : {m_bas:.2f} à {m_haut:.2f}")
assert abs(popt[1] + pente) < 1e-6, "curve_fit sur un modèle linéaire redonne polyfit"
assert abs(popt[1] - m_vrai) < 2 * erreurs[1], "la vraie valeur est dans l'intervalle"
# résidus : mesures moins modèle
residus = logN - wohler(S_MPa, popt[0], popt[1])
print(f"dispersion des résidus : {residus.std(ddof=2):.3f}")
r = pcov[0, 1] / (erreurs[0] * erreurs[1])   # corrélation des paramètres

In [ ]:
print(f"corrélation de log C et m : {r:.3f}")
dispersion_residus = float(residus.std(ddof=2))
assert abs(dispersion_residus - dispersion) < 0.03, "les résidus retrouvent le bruit simulé"
assert r > 0.99, "log C et m sont presque parfaitement corrélés"

## Exercice 12.4 : Exposant de Basquin et son incertitude (défi, 25 min)

Reprenez la campagne simulée (graine $42$, quinze essais) et ajustez la loi avec `curve_fit` : exposant $m$ et incertitude. Répétez la campagne $500$ fois, avec les graines $1000$ à $1499$ : l'écart-type des exposants ajustés doit retrouver l'incertitude annoncée par `pcov`.

**Résultat attendu.** $m = 3{,}2\pm 0{,}3$ ; sur $500$ campagnes, moyenne de 3,51 et écart-type de 0,30, pour une erreur annoncée de 0,29 en moyenne.

**Seconde voie.** La vraie valeur $3{,}5$ tombe dans l'intervalle de deux écarts-types pour 91 % des campagnes, un peu moins de $95\,\%$ (écart-type estimé).

**Indication.** Générez les essais avec `default_rng`, ajustez avec `curve_fit` et prenez `np.sqrt(pcov[1, 1])`. Pour les $500$ campagnes, une boucle `for` change la graine, refait l'ajustement et range $m$ et son erreur dans deux listes ; l'écart-type des $m$ (`ddof=1`) approche l'erreur moyenne, preuve que l'incertitude annoncée est la bonne.

Les contrôles de la cellule lisent les variables `couverture`, `ecart_type_m`, `erreur_moyenne`, `moyenne_m` : gardez ces noms.

In [ ]:
exposants, erreurs_m, dans_intervalle = [], [], 0
# Votre code :
assert abs(moyenne_m - m_vrai) < 0.05, "l'ajustement n'a pas de biais notable"
assert abs(ecart_type_m - erreur_moyenne) < 0.03, "l'écart-type des m égale l'erreur de pcov"
assert 88.0 <= couverture <= 97.0, "couverture attendue : un peu sous 95 %"
print("Exercice 12.4 : contrôles passés")

## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F12-1 : Racines d'une équation (`scipy.optimize.brentq`)

In [ ]:
import numpy as np
from scipy.optimize import brentq

L, S = 50.0, 52.0                           # portée et longueur, m

def ecart_longueur(a):
    return 2 * a * np.sinh(L / (2 * a)) - S # erreur de chaînette, m

a_eq = brentq(ecart_longueur, 10.0, 100.0)  # paramètre a, m
texte = f"a = {a_eq:.2f} m".replace(".", ",")
print(texte)
print(np.isclose(ecart_longueur(a_eq), 0.0))

### Fiche F12-2 : Minimisation (`scipy.optimize.minimize`)

In [ ]:
import numpy as np
from scipy.optimize import minimize

def energie(q):
    L1, L2 = np.hypot(q[0], q[1]), np.hypot(q[0] - 2.0, q[1])
    return 500.0 * ((L1 - 1.0)**2 + (L2 - 1.0)**2) + 500.0 * q[1]

res = minimize(energie, x0=[1.0, -0.5], method="BFGS")
print(f"Nœud : x = {res.x[0]:.3f} m, y = {res.x[1]:.3f} m".replace(".", ","))
print(np.isclose(res.x[0], 1.0))           # symétrie respectée

### Fiche F12-3 : Ajustement de modèle (`scipy.optimize.curve_fit`)

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

def hollomon(eps, K, n):
    return K * eps**n             # loi d'écrouissage

eps = np.linspace(0.01, 0.20, 20) # déformations plastiques
bruit = np.random.default_rng(0).normal(0.0, 5.0, 20)
sigma = 820.0 * eps**0.22 + bruit # contraintes mesurées, MPa

popt, pcov = curve_fit(hollomon, eps, sigma, p0=[800.0, 0.2])
K, n = popt                       # coefficient et exposant ajustés
print(f"K = {K:.0f} MPa, n = {n:.3f}".replace(".", ","))
print(np.allclose(sigma, hollomon(eps, K, n), atol=20.0)) # résidus